# 03 - EDA + PhÃ¢n tÃ­ch thá»‘ng kÃª + Quan há»‡ Ä‘áº·c trÆ°ng + Trá»±c quan hÃ³a
**NgÆ°á»i 2 â€” NhÃ¡nh:** `feature/visualization`

**Äáº§u vÃ o:** `data/processed/` (do NgÆ°á»i 1 táº¡o ra)  
**Biáº¿n má»¥c tiÃªu:** `formatted_experience_level`

## 0. Import thÆ° viá»‡n & CÃ i Ä‘áº·t chung

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, kruskal, f_oneway
from pathlib import Path
import warnings

# Táº¯t cÃ¡c cáº£nh bÃ¡o khÃ´ng cáº§n thiáº¿t
warnings.filterwarnings('ignore')

# CÃ i Ä‘áº·t hiá»ƒn thá»‹ báº£ng
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# CÃ i Ä‘áº·t Ä‘á»“ thá»‹
sns.set_style('whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# ÄÆ°á»ng dáº«n thÆ° má»¥c dá»¯ liá»‡u Ä‘Ã£ xá»­ lÃ½
PROCESSED = Path('../data/processed')

print('Import thÆ° viá»‡n thÃ nh cÃ´ng!')

## 1. Táº£i dá»¯ liá»‡u

In [ ]:
# Táº£i báº£ng chÃ­nh: postings_clean
df = pd.read_csv(PROCESSED / 'postings_clean.csv', low_memory=False)
print(f'postings_clean: {df.shape[0]:,} dÃ²ng x {df.shape[1]} cá»™t')

# HÃ m táº£i báº£ng phá»¥, kiá»ƒm tra tá»“n táº¡i trÆ°á»›c khi táº£i
def load(filename):
    path = PROCESSED / filename
    if path.exists():
        data = pd.read_csv(path)
        print(f'{filename}: {data.shape[0]:,} dÃ²ng x {data.shape[1]} cá»™t')
        return data
    print(f'KhÃ´ng tÃ¬m tháº¥y file: {filename}')
    return None

job_skills     = load('job_skills_clean.csv')
companies      = load('companies_clean.csv')
salaries       = load('salaries_clean.csv')
job_industries = load('job_industries_clean.csv')

In [ ]:
# Xem tá»•ng quan dá»¯ liá»‡u
print('Danh sÃ¡ch cá»™t trong postings_clean:')
print(list(df.columns))
df.head()

In [ ]:
# XÃ¡c Ä‘á»‹nh nhÃ³m cá»™t Ä‘á»ƒ phÃ¢n tÃ­ch
TARGET = 'formatted_experience_level'

# Cá»™t sá»‘: lÆ°Æ¡ng vÃ  má»©c Ä‘á»™ tÆ°Æ¡ng tÃ¡c
NUM_COLS = [c for c in ['min_salary', 'med_salary', 'max_salary',
                         'normalized_salary', 'views', 'applies']
            if c in df.columns]

# Cá»™t phÃ¢n loáº¡i
CAT_COLS = [c for c in ['formatted_work_type', 'work_type', 'remote_allowed',
                          'pay_period', 'compensation_type', 'currency']
            if c in df.columns]

# Cá»™t vÄƒn báº£n
TEXT_COLS = [c for c in ['title', 'description', 'skills_desc'] if c in df.columns]

print('Biáº¿n má»¥c tiÃªu :', TARGET,
      '| Thiáº¿u:', df[TARGET].isnull().sum(),
      f'({df[TARGET].isnull().mean()*100:.1f}%)')
print('Cá»™t sá»‘        :', NUM_COLS)
print('Cá»™t phÃ¢n loáº¡i :', CAT_COLS)
print('Cá»™t vÄƒn báº£n   :', TEXT_COLS)

In [ ]:
# Kiá»ƒm tra giÃ¡ trá»‹ thiáº¿u (chá»‰ hiá»ƒn thá»‹ cá»™t cÃ³ null)
missing = pd.DataFrame({
    'Kiá»ƒu dá»¯ liá»‡u'         : df.dtypes,
    'Sá»‘ null'              : df.isnull().sum(),
    '% null'               : (df.isnull().sum() / len(df) * 100).round(2),
    'Sá»‘ giÃ¡ trá»‹ khÃ¡c nhau': df.nunique()
})
missing[missing['Sá»‘ null'] > 0].sort_values('% null', ascending=False)

## 2. PhÃ¢n tÃ­ch Ä‘Æ¡n biáº¿n â€” Cá»™t sá»‘ (Univariate Numerical)
PhÃ¢n tÃ­ch tá»«ng cá»™t sá»‘: Trung bÃ¬nh, Trung vá»‹, Äá»™ lá»‡ch chuáº©n, IQR, Äá»™ lá»‡ch (Skewness), Ngoáº¡i lá»‡  
Biá»ƒu Ä‘á»“: Histogram + KDE + Boxplot

In [ ]:
# Thá»‘ng kÃª mÃ´ táº£ Ä‘áº§y Ä‘á»§ cho cÃ¡c cá»™t sá»‘
print('=== THá»NG KÃŠ MÃ” Táº¢ CÃC Cá»˜T Sá» ===')
desc = df[NUM_COLS].describe().T
desc['IQR']            = desc['75%'] - desc['25%']
desc['Äá»™ lá»‡ch (skew)'] = df[NUM_COLS].skew()
desc['Kurtosis']       = df[NUM_COLS].kurtosis()
desc.rename(columns={'25%': 'Q1', '50%': 'Trung vá»‹', '75%': 'Q3'}, inplace=True)
desc.round(2)

In [ ]:
# Váº½ Histogram + KDE + Boxplot cho tá»«ng cá»™t sá»‘
for col in NUM_COLS:
    data = df[col].dropna()
    if len(data) == 0:
        print(f'Cá»™t {col}: toÃ n bá»™ lÃ  NaN, bá» qua.')
        continue

    # TÃ­nh cÃ¡c chá»‰ sá»‘ phÃ¢n vá»‹ vÃ  ngoáº¡i lá»‡ theo phÆ°Æ¡ng phÃ¡p IQR
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR    = Q3 - Q1
    lower  = Q1 - 1.5 * IQR   # NgÆ°á»¡ng dÆ°á»›i
    upper  = Q3 + 1.5 * IQR   # NgÆ°á»¡ng trÃªn
    n_out  = ((data < lower) | (data > upper)).sum()   # Sá»‘ ngoáº¡i lá»‡

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # --- Biá»ƒu Ä‘á»“ Histogram + KDE ---
    axes[0].hist(data, bins=60, color='steelblue', edgecolor='white', alpha=0.7, density=True)
    data.plot.kde(ax=axes[0], color='crimson', linewidth=2)
    axes[0].axvline(data.mean(),   color='orange', linestyle='--', linewidth=1.5,
                    label=f'Trung bÃ¬nh = {data.mean():,.0f}')
    axes[0].axvline(data.median(), color='green',  linestyle='--', linewidth=1.5,
                    label=f'Trung vá»‹   = {data.median():,.0f}')
    axes[0].set_title(f'PhÃ¢n phá»‘i â€” {col}', fontsize=13, fontweight='bold')
    axes[0].legend()

    # Nháº­n xÃ©t Ä‘á»™ lá»‡ch phÃ¢n phá»‘i
    skew = data.skew()
    if skew > 0.5:
        skew_txt = 'Lá»‡ch pháº£i (dÆ°Æ¡ng)'
    elif skew < -0.5:
        skew_txt = 'Lá»‡ch trÃ¡i (Ã¢m)'
    else:
        skew_txt = 'Äá»‘i xá»©ng'
    axes[0].text(0.97, 0.95, f'Skew = {skew:.2f}\n({skew_txt})',
                 transform=axes[0].transAxes, ha='right', va='top', fontsize=9,
                 bbox=dict(facecolor='lightyellow', alpha=0.9, boxstyle='round'))

    # --- Biá»ƒu Ä‘á»“ Boxplot ---
    axes[1].boxplot(data, vert=False, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='.', color='gray', alpha=0.3, markersize=3))
    axes[1].set_title(f'Boxplot â€” {col}', fontsize=13, fontweight='bold')
    axes[1].text(0.97, 0.85,
                 f'Q1 = {Q1:,.0f}\nQ3 = {Q3:,.0f}\nIQR = {IQR:,.0f}\n'
                 f'Ngoáº¡i lá»‡: {n_out:,} ({n_out/len(data)*100:.1f}%)',
                 transform=axes[1].transAxes, ha='right', va='top', fontsize=9,
                 bbox=dict(facecolor='lightyellow', alpha=0.9, boxstyle='round'))

    plt.suptitle(f'PhÃ¢n tÃ­ch Ä‘Æ¡n biáº¿n: {col}  (n={len(data):,}, thiáº¿u={df[col].isnull().sum():,})',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. PhÃ¢n tÃ­ch Ä‘Æ¡n biáº¿n â€” Cá»™t phÃ¢n loáº¡i (Univariate Categorical)
Táº§n suáº¥t, Tá»· lá»‡ pháº§n trÄƒm, Cardinality  
Biá»ƒu Ä‘á»“: Cá»™t Ä‘áº¿m + Cá»™t tá»· lá»‡

In [ ]:
# Báº£ng táº§n suáº¥t cho tá»«ng cá»™t phÃ¢n loáº¡i
for col in CAT_COLS:
    vc  = df[col].value_counts(dropna=False)
    pct = df[col].value_counts(normalize=True, dropna=False) * 100
    # Chuyá»ƒn Ä‘á»•i nhÃ£n hiá»ƒn thá»‹, thay tháº¿ NaN thÃ nh '(Trá»‘ng)'
    lbls = [str(x) if pd.notna(x) else '(Trá»‘ng)' for x in vc.index]
    tbl = pd.DataFrame({'GiÃ¡ trá»‹': lbls, 'Sá»‘ lÆ°á»£ng': vc.values, 'Tá»· lá»‡ (%)': pct.round(2).values})
    print(f'\n{col}  | Cardinality={df[col].nunique()} | '
          f'Thiáº¿u={df[col].isnull().sum():,} ({df[col].isnull().mean()*100:.1f}%)')
    print(tbl.to_string(index=False))

In [ ]:
# Váº½ biá»ƒu Ä‘á»“ cá»™t cho tá»«ng cá»™t phÃ¢n loáº¡i (xá»­ lÃ½ an toÃ n nhÃ£n NaN)
for col in CAT_COLS:
    vc  = df[col].value_counts(dropna=False).head(15)
    pct = vc / len(df) * 100

    # Chuyá»ƒn nhÃ£n thÃ nh danh sÃ¡ch chuá»—i (str) thuáº§n tÃºy Ä‘á»ƒ trÃ¡nh lá»—i kiá»ƒu dá»¯ liá»‡u
    x_labels = [str(x) if pd.notna(x) else '(Trá»‘ng)' for x in vc.index]
    colors   = sns.color_palette('Set2', len(vc))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Biá»ƒu Ä‘á»“ sá»‘ lÆ°á»£ng
    bars = axes[0].bar(x_labels, vc.values, color=colors, edgecolor='white')
    for bar, p in zip(bars, pct.values):
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + vc.values.max() * 0.01,
                     f'{p:.1f}%', ha='center', va='bottom', fontsize=9)
    axes[0].set_title(f'Sá»‘ lÆ°á»£ng â€” {col}', fontsize=12, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=40)

    # Biá»ƒu Ä‘á»“ tá»· lá»‡ pháº§n trÄƒm
    axes[1].bar(x_labels, pct.values, color=colors, edgecolor='white')
    axes[1].set_ylabel('Tá»· lá»‡ (%)')
    axes[1].set_title(f'Tá»· lá»‡ pháº§n trÄƒm â€” {col}', fontsize=12, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=40)

    # ÄÆ°á»ng cÃ¢n báº±ng lÃ½ tÆ°á»Ÿng (náº¿u dá»¯ liá»‡u phÃ¢n bá»• Ä‘á»u giá»¯a cÃ¡c nhÃ³m)
    n_cat = df[col].nunique()
    if n_cat > 0:
        axes[1].axhline(100 / n_cat, color='red', linestyle='--', linewidth=1,
                        label=f'CÃ¢n báº±ng lÃ½ tÆ°á»Ÿng = {100/n_cat:.1f}%')
        axes[1].legend(fontsize=8)

    plt.suptitle(f'PhÃ¢n tÃ­ch Ä‘Æ¡n biáº¿n: {col}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Quan há»‡ Äáº·c trÆ°ng â€” Äáº·c trÆ°ng (Numerical vs Numerical)
TÆ°Æ¡ng quan Pearson & Spearman, Äa cá»™ng tuyáº¿n (Multicollinearity), Pair Plot

In [ ]:
# TÃ­nh ma tráº­n tÆ°Æ¡ng quan Pearson vÃ  Spearman
pearson  = df[NUM_COLS].corr(method='pearson')
spearman = df[NUM_COLS].corr(method='spearman')

# Chá»‰ hiá»ƒn thá»‹ ná»­a dÆ°á»›i cá»§a ma tráº­n Ä‘á»ƒ trÃ¡nh trÃ¹ng láº·p
mask = np.triu(np.ones_like(pearson, dtype=bool))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(pearson, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            ax=axes[0], linewidths=0.5, annot_kws={'size': 10})
axes[0].set_title('TÆ°Æ¡ng quan Pearson', fontsize=13, fontweight='bold')

sns.heatmap(spearman, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            ax=axes[1], linewidths=0.5, annot_kws={'size': 10})
axes[1].set_title('TÆ°Æ¡ng quan Spearman', fontsize=13, fontweight='bold')

plt.suptitle('Ma tráº­n tÆ°Æ¡ng quan giá»¯a cÃ¡c Ä‘áº·c trÆ°ng sá»‘', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Báº£ng phÃ¢n loáº¡i má»©c Ä‘á»™ tÆ°Æ¡ng quan tá»«ng cáº·p
rows = []
for i in range(len(NUM_COLS)):
    for j in range(i + 1, len(NUM_COLS)):
        r  = pearson.iloc[i, j]
        rs = spearman.iloc[i, j]
        a  = abs(r)
        # PhÃ¢n loáº¡i má»©c Ä‘á»™ tÆ°Æ¡ng quan
        if a > 0.7:
            strength = 'Máº¡nh (>0.7)'
        elif a > 0.4:
            strength = 'Trung bÃ¬nh (0.4â€“0.7)'
        else:
            strength = 'Yáº¿u (<0.4)'
        direction = 'DÆ°Æ¡ng (+)' if r > 0 else 'Ã‚m (âˆ’)'
        rows.append({
            'Äáº·c trÆ°ng 1' : NUM_COLS[i],
            'Äáº·c trÆ°ng 2' : NUM_COLS[j],
            'Pearson r'   : round(r, 3),
            'Spearman r'  : round(rs, 3),
            'Má»©c Ä‘á»™'      : strength,
            'Chiá»u'       : direction
        })

corr_df = pd.DataFrame(rows).sort_values('Pearson r', key=abs, ascending=False)
print('=== PHÃ‚N LOáº I Má»¨C Äá»˜ TÆ¯Æ NG QUAN ===')
print(corr_df.to_string(index=False))

# Kiá»ƒm tra Ä‘a cá»™ng tuyáº¿n
print('\n--- ÄA Cá»˜NG TUYáº¾N â€” MULTICOLLINEARITY (|r| > 0.7) ---')
multi = corr_df[corr_df['Pearson r'].abs() > 0.7]
if len(multi):
    print(multi[['Äáº·c trÆ°ng 1', 'Äáº·c trÆ°ng 2', 'Pearson r']].to_string(index=False))
    print('=> CÃ¡c cáº·p nÃ y mang thÃ´ng tin trÃ¹ng láº·p, cÃ¢n nháº¯c loáº¡i bá» khi modeling.')
else:
    print('KhÃ´ng phÃ¡t hiá»‡n Ä‘a cá»™ng tuyáº¿n nghiÃªm trá»ng.')

In [ ]:
# Váº½ Pair Plot Ä‘á»ƒ quan sÃ¡t quan há»‡ tá»«ng cáº·p Ä‘áº·c trÆ°ng sá»‘
# Chá»n cÃ¡c cá»™t cÃ³ sá»‘ lÆ°á»£ng dÃ²ng quan sÃ¡t chung Ä‘á»§ lá»›n
pair_cols = [c for c in ['normalized_salary', 'views', 'applies'] if c in df.columns]
sample_df = df[pair_cols].dropna()

if len(sample_df) > 0:
    n_sample = min(2000, len(sample_df))
    sample = sample_df.sample(n_sample, random_state=42)
    g = sns.pairplot(sample, diag_kind='kde',
                     plot_kws={'alpha': 0.25, 's': 15, 'color': 'steelblue'},
                     diag_kws={'color': 'steelblue', 'fill': True})
    g.fig.suptitle(f'Pair Plot â€” CÃ¡c Ä‘áº·c trÆ°ng sá»‘ (máº«u {n_sample:,} dÃ²ng)',
                   fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('KhÃ´ng cÃ³ Ä‘á»§ dá»¯ liá»‡u Ä‘á»“ng thá»i giá»¯a cÃ¡c cá»™t sá»‘ Ä‘á»ƒ váº½ Pair Plot.')

## 5. Quan há»‡ Äáº·c trÆ°ng â€” Má»¥c tiÃªu (Numerical vs Target)
So sÃ¡nh cÃ¡c cá»™t sá»‘ theo tá»«ng cáº¥p Ä‘á»™ kinh nghiá»‡m  
Kiá»ƒm Ä‘á»‹nh thá»‘ng kÃª: ANOVA vÃ  Kruskal-Wallis  
Biá»ƒu Ä‘á»“: Boxplot, Violin Plot

In [ ]:
# Thá»‘ng kÃª mÃ´ táº£ tá»«ng cá»™t sá»‘ theo cáº¥p Ä‘á»™ kinh nghiá»‡m
if TARGET in df.columns:
    for col in NUM_COLS:
        tbl = df.groupby(TARGET)[col].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
        print(f'\n=== {col.upper()} theo {TARGET} ===')
        print(tbl.round(2))

In [ ]:
# Váº½ Boxplot vÃ  Violin Plot theo tá»«ng Job Level
if TARGET in df.columns:
    for col in NUM_COLS:
        plot_df = df[[TARGET, col]].dropna()
        if len(plot_df) == 0:
            continue

        # Sáº¯p xáº¿p Job Level theo trung vá»‹ giáº£m dáº§n
        order = plot_df.groupby(TARGET)[col].median().sort_values(ascending=False).index

        fig, axes = plt.subplots(1, 2, figsize=(16, 5))

        # Boxplot
        sns.boxplot(data=plot_df, x=TARGET, y=col,
                    order=order, palette='Set2', ax=axes[0])
        axes[0].set_title(f'{col} theo Cáº¥p Ä‘á»™ â€” Boxplot', fontsize=12, fontweight='bold')
        axes[0].tick_params(axis='x', rotation=30)

        # Violin Plot (thá»ƒ hiá»‡n hÃ¬nh dáº¡ng phÃ¢n phá»‘i)
        sns.violinplot(data=plot_df, x=TARGET, y=col, order=order,
                       palette='Set2', ax=axes[1], inner='quartile', cut=0)
        axes[1].set_title(f'{col} theo Cáº¥p Ä‘á»™ â€” Violin', fontsize=12, fontweight='bold')
        axes[1].tick_params(axis='x', rotation=30)

        plt.suptitle(f'Quan há»‡ Äáº·c trÆ°ngâ€“Má»¥c tiÃªu: {col} vs {TARGET}',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

In [ ]:
# Kiá»ƒm Ä‘á»‹nh thá»‘ng kÃª: ANOVA vÃ  Kruskal-Wallis
if TARGET in df.columns:
    print('=== KIá»‚M Äá»ŠNH THá»NG KÃŠ (Î± = 0.05) ===')
    print('p < 0.05  â†’  Ä‘áº·c trÆ°ng cÃ³ quan há»‡ cÃ³ Ã½ nghÄ©a vá»›i biáº¿n má»¥c tiÃªu\n')

    results = []
    for col in NUM_COLS:
        # TÃ¡ch dá»¯ liá»‡u thÃ nh tá»«ng nhÃ³m theo Job Level
        groups = [df[df[TARGET] == g][col].dropna().values
                  for g in df[TARGET].dropna().unique()]
        groups = [g for g in groups if len(g) > 1]
        if len(groups) < 2:
            continue

        f_val, p_anova = f_oneway(*groups)   # Kiá»ƒm Ä‘á»‹nh ANOVA (giáº£ Ä‘á»‹nh phÃ¢n phá»‘i chuáº©n)
        h_val, p_kw    = kruskal(*groups)    # Kiá»ƒm Ä‘á»‹nh Kruskal-Wallis (khÃ´ng tham sá»‘)

        results.append({
            'Äáº·c trÆ°ng'       : col,
            'ANOVA F'         : round(f_val, 2),
            'ANOVA p'         : round(p_anova, 5),
            'Kruskal-Wallis H': round(h_val, 2),
            'KW p'            : round(p_kw, 5),
            'Káº¿t luáº­n'        : 'CÃ³ quan há»‡' if p_kw < 0.05 else 'KhÃ´ng rÃµ rÃ ng'
        })

    print(pd.DataFrame(results).to_string(index=False))
    print('\nLÆ°u Ã½: Æ¯u tiÃªn Kruskal-Wallis vÃ¬ dá»¯ liá»‡u salary/views bá»‹ lá»‡ch máº¡nh (skewed).')

## 6. PhÃ¢n tÃ­ch PhÃ¢n loáº¡i â€” Sá»‘ (Categorical vs Numerical)
LÆ°Æ¡ng, LÆ°á»£t xem, LÆ°á»£t á»©ng tuyá»ƒn theo Loáº¡i cÃ´ng viá»‡c, LÃ m tá»« xa, Chu ká»³ thanh toÃ¡n

In [ ]:
# Chá»n cá»™t lÆ°Æ¡ng Æ°u tiÃªn nháº¥t cÃ³ sáºµn trong dá»¯ liá»‡u
sal_col = next((c for c in ['normalized_salary', 'med_salary', 'min_salary']
                if c in df.columns), None)

# Táº¡o danh sÃ¡ch cÃ¡c cáº·p (cá»™t phÃ¢n loáº¡i, cá»™t sá»‘) cáº§n phÃ¢n tÃ­ch
pairs = []
for cat in ['formatted_work_type', 'remote_allowed', 'pay_period']:
    if cat in df.columns and sal_col:
        pairs.append((cat, sal_col))   # LÆ°Æ¡ng theo tá»«ng loáº¡i
for num in ['views', 'applies']:
    if 'formatted_work_type' in df.columns and num in df.columns:
        pairs.append(('formatted_work_type', num))   # LÆ°á»£t xem/á»©ng tuyá»ƒn theo loáº¡i cÃ´ng viá»‡c

# Váº½ Boxplot vÃ  biá»ƒu Ä‘á»“ trung bÃ¬nh
for cat_col, num_col in pairs:
    plot_df = df[[cat_col, num_col]].dropna()
    if len(plot_df) == 0:
        continue

    # Sáº¯p xáº¿p theo trung vá»‹ giáº£m dáº§n
    order = plot_df.groupby(cat_col)[num_col].median().sort_values(ascending=False).index

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Boxplot
    sns.boxplot(data=plot_df, x=cat_col, y=num_col,
                order=order, palette='Set2', ax=axes[0])
    axes[0].set_title(f'{num_col} theo {cat_col} â€” Boxplot', fontsize=11, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=30)

    # Biá»ƒu Ä‘á»“ cá»™t trung bÃ¬nh
    means = plot_df.groupby(cat_col)[num_col].mean().reindex(order)
    x_mean_labels = [str(x) for x in means.index]
    bars  = axes[1].bar(x_mean_labels, means.values,
                         color=sns.color_palette('Set2', len(means)), edgecolor='white')
    for bar, v in zip(bars, means.values):
        axes[1].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height(), f'{v:,.0f}',
                     ha='center', va='bottom', fontsize=9)
    axes[1].set_title(f'Trung bÃ¬nh {num_col} theo {cat_col}', fontsize=11, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=30)

    plt.suptitle(f'PhÃ¢n loáº¡i vs Sá»‘: {cat_col} Ã— {num_col}', fontsize=13)
    plt.tight_layout()
    plt.show()

## 7. PhÃ¢n tÃ­ch PhÃ¢n loáº¡i â€” PhÃ¢n loáº¡i (Categorical vs Categorical)
Báº£ng chÃ©o (Crosstab), Kiá»ƒm Ä‘á»‹nh Chi-square, Biá»ƒu Ä‘á»“ cá»™t chá»“ng (Stacked Bar)

In [ ]:
if TARGET in df.columns:
    # Danh sÃ¡ch cá»™t phÃ¢n loáº¡i cáº§n so sÃ¡nh vá»›i biáº¿n má»¥c tiÃªu
    cat_list     = [c for c in ['formatted_work_type', 'remote_allowed', 'pay_period']
                    if c in df.columns]
    chi2_results = []

    for col2 in cat_list:
        plot_df = df[[TARGET, col2]].dropna()
        if len(plot_df) == 0:
            continue

        # Táº¡o báº£ng chÃ©o sá»‘ lÆ°á»£ng vÃ  tá»· lá»‡ pháº§n trÄƒm
        ct     = pd.crosstab(plot_df[TARGET], plot_df[col2])
        ct_pct = pd.crosstab(plot_df[TARGET], plot_df[col2], normalize='index') * 100

        print(f'\n=== Báº¢NG CHÃ‰O: {TARGET} Ã— {col2} ===')
        print(ct)
        print('\nTá»· lá»‡ % theo hÃ ng:')
        print(ct_pct.round(1))

        # Kiá»ƒm Ä‘á»‹nh Chi-square: kiá»ƒm tra sá»± Ä‘á»™c láº­p giá»¯a 2 biáº¿n phÃ¢n loáº¡i
        chi2, p, dof, _ = chi2_contingency(ct)
        chi2_results.append({
            'Cáº·p biáº¿n'  : f'{TARGET} Ã— {col2}',
            'Chi2'      : round(chi2, 2),
            'p-value'   : round(p, 5),
            'Báº­c tá»± do' : dof,
            'Ã nghÄ©a?'  : 'CÃ“' if p < 0.05 else 'KHÃ”NG'
        })

        # Biá»ƒu Ä‘á»“ cá»™t chá»“ng theo tá»· lá»‡
        ct_pct.plot(kind='bar', stacked=True, figsize=(12, 5),
                    colormap='Set2', edgecolor='white', linewidth=0.5)
        plt.title(f'Biá»ƒu Ä‘á»“ cá»™t chá»“ng: {TARGET} Ã— {col2}'
                  f'\n(Chi2={chi2:.1f}, p={p:.4f})',
                  fontsize=12, fontweight='bold')
        plt.ylabel('Tá»· lá»‡ (%)')
        plt.xticks(rotation=30)
        plt.legend(title=col2, bbox_to_anchor=(1.05, 1))
        plt.tight_layout()
        plt.show()

    print('\n=== TÃ“M Táº®T KIá»‚M Äá»ŠNH CHI-SQUARE ===')
    print(pd.DataFrame(chi2_results).to_string(index=False))

## 8. PhÃ¢n tÃ­ch Äa biáº¿n (Multivariate Analysis)
LÆ°Æ¡ng Ã— Cáº¥p Ä‘á»™ Ã— Loáº¡i cÃ´ng viá»‡c + Heatmap tÆ°Æ¡ng quan toÃ n bá»™

In [ ]:
# PhÃ¢n tÃ­ch 3 chiá»u: LÆ°Æ¡ng Ã— Cáº¥p Ä‘á»™ kinh nghiá»‡m Ã— Loáº¡i cÃ´ng viá»‡c
sal_col  = next((c for c in ['normalized_salary', 'med_salary'] if c in df.columns), None)
work_col = 'formatted_work_type'

if TARGET in df.columns and sal_col and work_col in df.columns:
    plot_df = df[[TARGET, sal_col, work_col]].dropna()
    order   = plot_df.groupby(TARGET)[sal_col].median().sort_values().index

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.boxplot(data=plot_df, x=TARGET, y=sal_col,
                hue=work_col, order=order, palette='Set2', ax=ax)
    ax.set_title(f'PhÃ¢n tÃ­ch Ä‘a biáº¿n: {sal_col} Ã— {TARGET} Ã— {work_col}',
                 fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Loáº¡i cÃ´ng viá»‡c', bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.show()

In [ ]:
# Heatmap tÆ°Æ¡ng quan Ä‘áº§y Ä‘á»§: cá»™t sá»‘ + mÃ£ hÃ³a sá»‘ cá»™t phÃ¢n loáº¡i
df_enc = df[NUM_COLS].copy()
for col in CAT_COLS:
    if col in df.columns and df[col].nunique() <= 10:
        # MÃ£ hÃ³a sá»‘ cho cá»™t phÃ¢n loáº¡i cÃ³ Ã­t nhÃ£n (â‰¤ 10 giÃ¡ trá»‹)
        df_enc[col + '_code'] = df[col].astype('category').cat.codes

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(df_enc.corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.3, ax=ax, annot_kws={'size': 8})
ax.set_title('Heatmap tÆ°Æ¡ng quan Ä‘áº§y Ä‘á»§ (Sá»‘ + PhÃ¢n loáº¡i Ä‘Ã£ mÃ£ hÃ³a)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. PhÃ¢n tÃ­ch Ngoáº¡i lá»‡ (Outlier Analysis)
PhÆ°Æ¡ng phÃ¡p IQR & Z-score â€” PhÃ¢n biá»‡t lá»—i dá»¯ liá»‡u vs giÃ¡ trá»‹ cá»±c trá»‹ há»£p lá»‡

In [ ]:
# Báº£ng tá»•ng há»£p ngoáº¡i lá»‡ cho táº¥t cáº£ cá»™t sá»‘
print('=== PHÃ‚N TÃCH NGOáº I Lá»† ===')
outlier_rows = []
for col in NUM_COLS:
    data = df[col].dropna()
    if len(data) == 0:
        continue

    # PhÆ°Æ¡ng phÃ¡p IQR
    Q1, Q3  = data.quantile(0.25), data.quantile(0.75)
    IQR     = Q3 - Q1
    lower   = Q1 - 1.5 * IQR   # NgÆ°á»¡ng dÆ°á»›i
    upper   = Q3 + 1.5 * IQR   # NgÆ°á»¡ng trÃªn
    n_iqr   = ((data < lower) | (data > upper)).sum()

    # PhÆ°Æ¡ng phÃ¡p Z-score (ngÆ°á»¡ng 3 Ä‘á»™ lá»‡ch chuáº©n)
    n_z = (np.abs(stats.zscore(data)) > 3).sum()

    outlier_rows.append({
        'Äáº·c trÆ°ng'       : col,
        'n'               : len(data),
        'Ngoáº¡i lá»‡ IQR'    : n_iqr,
        'IQR %'           : round(n_iqr / len(data) * 100, 2),
        'Ngoáº¡i lá»‡ Z>3Ïƒ'   : n_z,
        'Z %'             : round(n_z / len(data) * 100, 2),
        'NgÆ°á»¡ng dÆ°á»›i IQR' : round(lower, 1),
        'NgÆ°á»¡ng trÃªn IQR' : round(upper, 1)
    })

print(pd.DataFrame(outlier_rows).to_string(index=False))
print('\nNháº­n xÃ©t: Ngoáº¡i lá»‡ lÆ°Æ¡ng = GiÃ¡ trá»‹ cá»±c trá»‹ há»£p lá»‡ (lÆ°Æ¡ng Director/Executive)')
print('=> KhÃ´ng xÃ³a. NÃªn dÃ¹ng Log Transform khi modeling Ä‘á»ƒ giáº£m áº£nh hÆ°á»Ÿng.')

In [ ]:
# So sÃ¡nh phÃ¢n phá»‘i gá»‘c vÃ  sau Log Transform cho cá»™t lÆ°Æ¡ng
sal_col = next((c for c in ['normalized_salary', 'min_salary'] if c in df.columns), None)
if sal_col:
    data   = df[sal_col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR    = Q3 - Q1
    lower  = Q1 - 1.5 * IQR
    upper  = Q3 + 1.5 * IQR
    n_out  = ((data < lower) | (data > upper)).sum()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Boxplot Ä‘á»ƒ tháº¥y vá»‹ trÃ­ ngoáº¡i lá»‡
    axes[0].boxplot(data, vert=False, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='.', color='gray', alpha=0.2, markersize=3))
    axes[0].set_title(f'Boxplot: {sal_col}\n'
                      f'Ngoáº¡i lá»‡ = {n_out:,} ({n_out/len(data)*100:.1f}%)',
                      fontsize=11, fontweight='bold')

    # PhÃ¢n phá»‘i gá»‘c (bá»‹ lá»‡ch máº¡nh)
    axes[1].hist(data, bins=60, color='steelblue', alpha=0.7, edgecolor='white', density=True)
    axes[1].set_title(f'PhÃ¢n phá»‘i gá»‘c\nÄá»™ lá»‡ch (Skew) = {data.skew():.2f}',
                      fontsize=11, fontweight='bold')

    # Sau Log Transform (phÃ¢n phá»‘i gáº§n chuáº©n hÆ¡n)
    log_data = np.log1p(data[data > 0])
    axes[2].hist(log_data, bins=60, color='seagreen', alpha=0.7, edgecolor='white', density=True)
    axes[2].set_title(f'Sau Log(1+x)\nÄá»™ lá»‡ch (Skew) = {log_data.skew():.2f}',
                      fontsize=11, fontweight='bold')
    axes[2].set_xlabel('log(1 + lÆ°Æ¡ng)')

    plt.suptitle(f'Ngoáº¡i lá»‡ & Log Transform â€” {sal_col}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 10. PhÃ¢n tÃ­ch máº¥t cÃ¢n báº±ng lá»›p (Class Imbalance)
Biáº¿n má»¥c tiÃªu: `formatted_experience_level` â†’ BÃ n giao káº¿t quáº£ cho NgÆ°á»i 4

In [ ]:
if TARGET in df.columns:
    # Äáº¿m sá»‘ lÆ°á»£ng vÃ  tá»· lá»‡ tá»«ng lá»›p (ká»ƒ cáº£ NaN)
    counts = df[TARGET].value_counts(dropna=False)
    pcts   = df[TARGET].value_counts(normalize=True, dropna=False) * 100

    # Chuyá»ƒn nhÃ£n an toÃ n, khÃ´ng Ä‘á»ƒ NaN dáº¡ng float
    target_labels = [str(x) if pd.notna(x) else '(Trá»‘ng / NaN)' for x in counts.index]

    print('=== PHÃ‚N PHá»I CÃC Lá»šP â€” formatted_experience_level ===')
    tbl_imbalance = pd.DataFrame({'Lá»›p': target_labels, 'Sá»‘ lÆ°á»£ng': counts.values, 'Tá»· lá»‡ (%)': pcts.round(2).values})
    print(tbl_imbalance.to_string(index=False))

    # TÃ­nh tá»· lá»‡ máº¥t cÃ¢n báº±ng (lá»›p lá»›n nháº¥t / lá»›p bÃ© nháº¥t khÃ´ng tÃ­nh NaN)
    valid_counts = df[TARGET].dropna().value_counts()
    if len(valid_counts) > 1:
        ratio = valid_counts.max() / valid_counts.min()
        print(f'\nTá»· lá»‡ máº¥t cÃ¢n báº±ng (lá»›p lá»›n nháº¥t / lá»›p bÃ© nháº¥t) = {ratio:.1f} : 1')
        if ratio > 5:
            print('Máº¤T CÃ‚N Báº°NG NGHIÃŠM TRá»ŒNG')
            print('=> NgÆ°á»i 4 cáº§n xá»­ lÃ½: SMOTE / class_weight / stratified split')
        elif ratio > 2:
            print('Máº¤T CÃ‚N Báº°NG Vá»ªA')
            print('=> NÃªn dÃ¹ng class_weight khi huáº¥n luyá»‡n mÃ´ hÃ¬nh')
        else:
            print('TÆ°Æ¡ng Ä‘á»‘i cÃ¢n báº±ng')

    colors = sns.color_palette('Set2', len(counts))
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Biá»ƒu Ä‘á»“ cá»™t
    bars = axes[0].bar(target_labels, counts.values, color=colors, edgecolor='white')
    for bar, cnt, pct in zip(bars, counts.values, pcts.values):
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + counts.values.max() * 0.01,
                     f'{cnt:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
    axes[0].set_title('PhÃ¢n phá»‘i lá»›p â€” Cáº¥p Ä‘á»™ kinh nghiá»‡m', fontsize=13, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=30)

    # Biá»ƒu Ä‘á»“ trÃ²n
    axes[1].pie(counts.values,
                labels=[f'{lbl}\n({p:.1f}%)' for lbl, p in zip(target_labels, pcts.values)],
                colors=colors, startangle=90,
                wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
    axes[1].set_title('Tá»· lá»‡ pháº§n trÄƒm â€” Cáº¥p Ä‘á»™ kinh nghiá»‡m', fontsize=13, fontweight='bold')

    plt.suptitle('PhÃ¢n tÃ­ch máº¥t cÃ¢n báº±ng lá»›p  â†’  BÃ n giao NgÆ°á»i 4',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 11. PhÃ¢n tÃ­ch Ká»¹ nÄƒng & VÄƒn báº£n (Skill & Text Analysis)
Äá»™ dÃ i vÄƒn báº£n, Sá»‘ tá»«, Tá»« khÃ³a phá»• biáº¿n, Ká»¹ nÄƒng theo Cáº¥p Ä‘á»™

In [ ]:
# Táº¡o Ä‘áº·c trÆ°ng tá»« cá»™t vÄƒn báº£n
if 'title' in df.columns:
    df['title_length']     = df['title'].fillna('').str.len()              # Äá»™ dÃ i tiÃªu Ä‘á» (kÃ½ tá»±)
    df['title_word_count'] = df['title'].fillna('').str.split().str.len()  # Sá»‘ tá»« trong tiÃªu Ä‘á»
if 'description' in df.columns:
    df['desc_length']      = df['description'].fillna('').str.len()        # Äá»™ dÃ i mÃ´ táº£
    df['desc_word_count']  = df['description'].fillna('').str.split().str.len()  # Sá»‘ tá»« trong mÃ´ táº£

text_feat = [c for c in ['title_length', 'title_word_count', 'desc_length', 'desc_word_count']
             if c in df.columns]
if text_feat:
    print('=== THá»NG KÃŠ Äáº¶C TRÆ¯NG VÄ‚N Báº¢N ===')
    print(df[text_feat].describe().round(2))

In [ ]:
# So sÃ¡nh Ä‘á»™ dÃ i vÄƒn báº£n theo tá»«ng Cáº¥p Ä‘á»™ kinh nghiá»‡m
if TARGET in df.columns:
    for col in ['desc_length', 'title_length']:
        if col not in df.columns:
            continue
        plot_df = df[[TARGET, col]].dropna()
        order   = plot_df.groupby(TARGET)[col].median().sort_values(ascending=False).index

        fig, ax = plt.subplots(figsize=(12, 5))
        sns.boxplot(data=plot_df, x=TARGET, y=col,
                    order=order, palette='Set2', ax=ax)
        ax.set_title(f'{col} theo Cáº¥p Ä‘á»™ kinh nghiá»‡m', fontsize=12, fontweight='bold')
        ax.tick_params(axis='x', rotation=30)
        plt.tight_layout()
        plt.show()

In [ ]:
# PhÃ¢n tÃ­ch tá»« khÃ³a phá»• biáº¿n trong mÃ´ táº£ cÃ´ng viá»‡c
if 'description' in df.columns:
    # Danh sÃ¡ch tá»« dá»«ng (stop words) cáº§n loáº¡i bá»
    stop_words = {'the','and','or','in','of','to','a','for','is','are','with','on',
                  'at','be','as','an','we','our','you','will','this','that','have',
                  'it','from','by','was','not','your','can','has','all','they','their',
                  'work','role','team','job','experience','position','skills','ability'}

    # TÃ¡ch tá»«, loáº¡i stop words, Ä‘áº¿m táº§n suáº¥t
    all_words = (df['description'].fillna('')
                   .str.lower()
                   .str.replace(r'[^a-z\s]', ' ', regex=True)
                   .str.split().explode())
    top_keywords = (all_words[(all_words.str.len() > 3) & (~all_words.isin(stop_words))]
                      .value_counts().head(25))

    kw_labels = [str(x) for x in top_keywords.index]
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(kw_labels, top_keywords.values,
           color=sns.color_palette('viridis', 25), edgecolor='white')
    ax.set_title('Top 25 Tá»« khÃ³a phá»• biáº¿n trong MÃ´ táº£ CÃ´ng viá»‡c',
                 fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Top ká»¹ nÄƒng vÃ  phÃ¢n tÃ­ch ká»¹ nÄƒng theo Cáº¥p Ä‘á»™
if job_skills is not None:
    skill_col = next((c for c in ['skill_abr', 'skill_name', 'skill']
                      if c in job_skills.columns), None)
    if skill_col:
        # Top 20 ká»¹ nÄƒng phá»• biáº¿n nháº¥t trong toÃ n dataset
        top20 = job_skills[skill_col].value_counts().head(20)
        skill_labels = [str(x) if pd.notna(x) else '(Trá»‘ng)' for x in top20.index]

        fig, ax = plt.subplots(figsize=(14, 6))
        ax.bar(skill_labels, top20.values,
               color=sns.color_palette('Set2', len(top20)), edgecolor='white')
        ax.set_title('Top 20 Ká»¹ nÄƒng phá»• biáº¿n nháº¥t (job_skills_clean.csv)',
                     fontsize=13, fontweight='bold')
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()

        # PhÃ¢n tÃ­ch ká»¹ nÄƒng theo tá»«ng Cáº¥p Ä‘á»™ kinh nghiá»‡m
        if TARGET in df.columns:
            id_col = next((c for c in ['job_id', 'jobId']
                           if c in job_skills.columns and c in df.columns), None)
            if id_col:
                # GhÃ©p báº£ng ká»¹ nÄƒng vá»›i báº£ng chÃ­nh qua job_id
                merged = job_skills.merge(
                    df[[id_col, TARGET]].dropna(), on=id_col, how='inner')
                print('\n=== TOP 5 Ká»¸ NÄ‚NG THEO Cáº¤P Äá»˜ KINH NGHIá»†M ===')
                for level in sorted(merged[TARGET].unique()):
                    top5 = merged[merged[TARGET] == level][skill_col].value_counts().head(5)
                    print(f'\n  {level}:')
                    print(top5.to_string())

## 12. Nháº­n xÃ©t nghiá»‡p vá»¥ & Dashboard tá»•ng káº¿t (Business Insights)

In [ ]:
print('=' * 65)
print('  NHáº¬N XÃ‰T NGHIá»†P Vá»¤ â€” DS JOB RECOMMEND')
print('=' * 65)

if TARGET in df.columns:
    counts = df[TARGET].value_counts()
    print(f'\n1. Cáº¥p Ä‘á»™ tuyá»ƒn dá»¥ng nhiá»u nháº¥t: {counts.idxmax()} ({counts.max():,} tin)')
    print(f'   Cáº¥p Ä‘á»™ tuyá»ƒn dá»¥ng Ã­t nháº¥t:    {counts.idxmin()} ({counts.min():,} tin)')

sal_col = next((c for c in ['normalized_salary', 'med_salary'] if c in df.columns), None)
if sal_col and TARGET in df.columns:
    by_level = df.groupby(TARGET)[sal_col].median().sort_values(ascending=False)
    print(f'\n2. LÆ°Æ¡ng trung vá»‹ (median) theo cáº¥p Ä‘á»™ ({sal_col}):')
    for level, val in by_level.items():
        print(f'   {level}: ${val:,.0f}')

if 'corr_df' in dir() and len(corr_df):
    multi = corr_df[corr_df['Pearson r'].abs() > 0.7]
    print(f'\n3. Äa cá»™ng tuyáº¿n (|r| > 0.7):')
    if len(multi):
        for _, row in multi.iterrows():
            print(f'   {row["Äáº·c trÆ°ng 1"]} <-> {row["Äáº·c trÆ°ng 2"]}: r={row["Pearson r"]}')
    else:
        print('   KhÃ´ng phÃ¡t hiá»‡n Ä‘a cá»™ng tuyáº¿n nghiÃªm trá»ng.')

print('\nBÃ n giao káº¿t quáº£:')
print('   â†’ NgÆ°á»i 3 (Feature Engineering): log_salary, salary_range, text_length features')
print('   â†’ NgÆ°á»i 4 (Modeling): tá»· lá»‡ máº¥t cÃ¢n báº±ng lá»›p, danh sÃ¡ch Ä‘áº·c trÆ°ng cÃ³ quan há»‡')
print('=' * 65)

In [ ]:
# Dashboard tá»•ng káº¿t EDA â€” 4 biá»ƒu Ä‘á»“ chÃ­nh
sal_col = next((c for c in ['normalized_salary', 'med_salary'] if c in df.columns), None)

if sal_col and TARGET in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    palette = sns.color_palette('Set2')

    # --- Ã” 1: PhÃ¢n phá»‘i cáº¥p Ä‘á»™ kinh nghiá»‡m (bá» qua NaN Ä‘á»ƒ biá»ƒu Ä‘á»“ rÃµ rÃ ng) ---
    valid_target = df[TARGET].dropna().value_counts()
    target_x = [str(x) for x in valid_target.index]
    axes[0, 0].bar(target_x, valid_target.values,
                   color=palette[:len(valid_target)], edgecolor='white')
    for i, (cnt, pct) in enumerate(zip(valid_target.values, valid_target / valid_target.sum() * 100)):
        axes[0, 0].text(i, cnt + valid_target.max() * 0.01,
                        f'{pct:.0f}%', ha='center', fontsize=9)
    axes[0, 0].set_title('PhÃ¢n phá»‘i cáº¥p Ä‘á»™ kinh nghiá»‡m', fontweight='bold')
    axes[0, 0].tick_params(axis='x', rotation=30)

    # --- Ã” 2: LÆ°Æ¡ng theo cáº¥p Ä‘á»™ kinh nghiá»‡m ---
    order = df.groupby(TARGET)[sal_col].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=TARGET, y=sal_col,
                order=order, palette='Set2', ax=axes[0, 1])
    axes[0, 1].set_title(f'{sal_col} theo Cáº¥p Ä‘á»™ kinh nghiá»‡m', fontweight='bold')
    axes[0, 1].tick_params(axis='x', rotation=30)

    # --- Ã” 3: TÆ°Æ¡ng quan giá»¯a cÃ¡c cá»™t lÆ°Æ¡ng ---
    sal_cols = [c for c in ['min_salary', 'med_salary', 'max_salary', 'normalized_salary']
                if c in df.columns]
    if len(sal_cols) >= 2:
        sns.heatmap(df[sal_cols].corr(), annot=True, fmt='.2f', cmap='RdYlGn',
                    center=0, ax=axes[1, 0], linewidths=0.5, annot_kws={'size': 10})
        axes[1, 0].set_title('TÆ°Æ¡ng quan giá»¯a cÃ¡c cá»™t lÆ°Æ¡ng', fontweight='bold')

    # --- Ã” 4: Quan há»‡ LÆ°á»£t xem vs LÆ°á»£t á»©ng tuyá»ƒn ---
    if 'views' in df.columns and 'applies' in df.columns:
        va_df = df[['views', 'applies']].dropna()
        if len(va_df) > 0:
            sample = va_df.sample(min(3000, len(va_df)), random_state=42)
            axes[1, 1].scatter(sample['views'], sample['applies'],
                               alpha=0.3, s=10, color='steelblue')
            r = va_df.corr().iloc[0, 1]
            axes[1, 1].set_title(f'LÆ°á»£t xem vs LÆ°á»£t á»©ng tuyá»ƒn  (r={r:.2f})', fontweight='bold')
            axes[1, 1].set_xlabel('LÆ°á»£t xem (Views)')
            axes[1, 1].set_ylabel('LÆ°á»£t á»©ng tuyá»ƒn (Applies)')

    plt.suptitle('Dashboard Tá»•ng káº¿t EDA â€” DS Job Recommend',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('\nPhÃ¢n tÃ­ch EDA hoÃ n thÃ nh!')